# 02. Bayesian Modeling: Prior, Likelihood, Posterior

Bayesian modeling은 파라미터를 고정된 미지수가 아니라 확률분포로 표현한다.

$$p(\theta\mid D)=\frac{p(D\mid\theta)p(\theta)}{p(D)}$$

로보틱스에서는 데이터가 적거나 노이즈가 큰 상황이 많기 때문에, prior와 posterior를 명시적으로 다루는 관점이 유용하다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. Beta-Bernoulli: 충돌 확률 추정

어떤 좁은 passage를 통과할 때 성공/실패 데이터가 쌓인다고 하자. 성공 확률 $\theta$에 대한 prior를 Beta distribution으로 두면 posterior도 Beta distribution이 된다.

$$\theta \sim Beta(\alpha,\beta), \qquad y_i\sim Bernoulli(\theta)$$

$$p(\theta\mid y)=Beta(\alpha+s,\beta+f)$$

In [ ]:
import math

def beta_pdf(x, a, b):
    coef = math.gamma(a + b) / (math.gamma(a) * math.gamma(b))
    return coef * (x ** (a - 1)) * ((1 - x) ** (b - 1))

x = np.linspace(0.001, 0.999, 400)
alpha0, beta0 = 2, 2
trials = np.array([1, 0, 1, 1, 0, 1, 1, 1])  # 1 = safe traversal, 0 = collision/failure
success = int(trials.sum())
failure = len(trials) - success
alpha1, beta1 = alpha0 + success, beta0 + failure

prior = np.array([beta_pdf(v, alpha0, beta0) for v in x])
posterior = np.array([beta_pdf(v, alpha1, beta1) for v in x])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, prior, lw=2, label=f'prior Beta({alpha0},{beta0})')
ax.plot(x, posterior, lw=2, label=f'posterior Beta({alpha1},{beta1})')
ax.axvline(alpha1 / (alpha1 + beta1), color='tab:red', ls='--', label='posterior mean')
ax.set_title('Bayesian update for traversal success probability')
ax.set_xlabel('success probability $\\theta$')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_02_beta_bernoulli_update.png', dpi=160)
plt.show()

print('success/failure:', success, failure)
print('posterior mean:', round(alpha1 / (alpha1 + beta1), 3))

## 2. Normal-Normal: 센서 Bias Bayesian Calibration

Range sensor bias $b$를 추정한다고 하자. prior와 likelihood가 Gaussian이면 posterior도 Gaussian이다.

$$b\sim\mathcal{N}(\mu_0,\sigma_0^2), \qquad z_i-d_i\sim\mathcal{N}(b,\sigma^2)$$

In [ ]:
np.random.seed(502)
true_bias = 0.22
known_sigma = 0.18
residuals = true_bias + np.random.randn(12) * known_sigma

mu0 = 0.0
sigma0 = 0.35
n = len(residuals)
sample_mean = residuals.mean()
posterior_var = 1 / (1 / sigma0**2 + n / known_sigma**2)
posterior_mu = posterior_var * (mu0 / sigma0**2 + n * sample_mean / known_sigma**2)
posterior_sigma = np.sqrt(posterior_var)

xs = np.linspace(-0.45, 0.65, 400)
def normal_pdf(x, mu, sig):
    return 1 / (sig * np.sqrt(2*np.pi)) * np.exp(-0.5 * ((x - mu) / sig) ** 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(xs, normal_pdf(xs, mu0, sigma0), lw=2, label='prior')
ax.plot(xs, normal_pdf(xs, posterior_mu, posterior_sigma), lw=2, label='posterior')
ax.scatter(residuals, np.zeros_like(residuals), color='black', s=25, label='bias samples')
ax.axvline(true_bias, color='tab:green', label='true bias')
ax.set_title('Bayesian sensor bias calibration')
ax.set_xlabel('bias [m]')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_02_sensor_bias_posterior.png', dpi=160)
plt.show()

print('sample mean:', round(sample_mean, 3))
print('posterior mean:', round(posterior_mu, 3))
print('posterior sigma:', round(posterior_sigma, 3))

## 3. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Prior | 데이터 전의 믿음 | 적은 데이터에서 보수적 추정 |
| Likelihood | 파라미터가 데이터를 설명하는 정도 | sensor model / dynamics model |
| Posterior | 데이터 반영 후 믿음 | calibration, risk estimation |
| Conjugate model | posterior 계산이 닫힌 형태 | 빠른 online update |

Bayesian modeling은 단순히 평균 하나를 찾는 것이 아니라, **얼마나 확신하는지**까지 같이 추정한다.